# Task 4: Forecasting Access and Usage (2025-2027)

Targets:
1. **Access** — Account Ownership Rate (`ACC_OWNERSHIP`)
2. **Usage** — Digital Payment Adoption (`USG_DIGITAL_PAYMENT`)

Approaches: OLS trend, event-augmented trend, optimistic/base/pessimistic scenarios.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.express as px

from src.data_loader import load_unified_data, load_impact_sheet
from src.impact_model import join_events_impacts, cumulative_event_effects
from src.forecast import build_forecast_bundle

FIG = ROOT / 'reports' / 'figures'
MODEL = ROOT / 'models'

data = load_unified_data()
impacts = load_impact_sheet()
obs = data[data.record_type=='observation'].copy()
joined = join_events_impacts(data, impacts)

years = list(range(2011, 2028))
access_eff, _ = cumulative_event_effects(joined, 'ACC_OWNERSHIP', years)
usage_eff, _ = cumulative_event_effects(joined, 'USG_DIGITAL_PAYMENT', years, form='linear_ramp')

bundle = build_forecast_bundle(obs, access_eff, usage_eff, [2025,2026,2027])
summary = bundle['summary']
print(summary)
summary.to_csv(MODEL/'forecast_summary.csv', index=False)
bundle['access_scenarios'].to_csv(MODEL/'access_scenarios.csv', index=False)
bundle['usage_scenarios'].to_csv(MODEL/'usage_scenarios.csv', index=False)


In [ ]:
acc_hist = bundle['access_history'].rename(columns={'value_numeric':'value'})
acc_hist['scenario'] = 'history'
acc_sc = bundle['access_scenarios'][['year','scenario','value']]
acc_plot = pd.concat([acc_hist[['year','scenario','value']], acc_sc], ignore_index=True)
fig = px.line(acc_plot, x='year', y='value', color='scenario', markers=True,
              title='Access scenarios (Account Ownership %)')
fig.show()
fig.write_html(FIG/'access_scenarios.html')

us_hist = bundle['usage_history'].rename(columns={'value_numeric':'value'})
us_hist['scenario'] = 'history'
us_sc = bundle['usage_scenarios'][['year','scenario','value']]
us_plot = pd.concat([us_hist[['year','scenario','value']], us_sc], ignore_index=True)
fig2 = px.line(us_plot, x='year', y='value', color='scenario', markers=True,
               title='Usage scenarios (Digital Payment Adoption %)')
fig2.show()
fig2.write_html(FIG/'usage_scenarios.html')

aug = bundle['access_event_augmented']
fig3 = px.line(aug, x='year', y='point', title='Access forecast with intervals')
fig3.add_scatter(x=aug.year, y=aug.upper, mode='lines', name='upper', line=dict(dash='dot'))
fig3.add_scatter(x=aug.year, y=aug.lower, mode='lines', name='lower', line=dict(dash='dot'))
fig3.show()


## Interpretation
- **Base Access** continues gradual gains but stays below NFIS-II stretch ambitions without rural breakthroughs.
- **Usage** has more upside from interoperability, merchant acceptance, and active-user conversion.
- Largest upside levers: interoperability (EthSwitch), merchant QR expansion, narrowing registered-active gap.
- Key uncertainties: Findex sparsity, impact scaling, macro/FX affordability shocks, definition mismatch operator vs survey.

See `reports/forecast_interpretation.md`.
